<a href="https://www.kaggle.com/code/fareedhassan/diabetes-predictions?scriptVersionId=366858490" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Explainable Diabetes Risk Prediction
Pima Indians dataset · logistic regression vs random forest · SHAP explanations.

**Setup:** click *Add Input* and add the 'Pima Indians Diabetes Database' dataset (or upload `diabetes_pima_real.csv` from the data pack).

In [ ]:
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, precision_recall_curve
from sklearn.calibration import CalibrationDisplay
SEED = 42
import os
cands = glob.glob('/kaggle/input/**/*.csv', recursive=True) + glob.glob('*.csv')
print('CSV files found:', cands)
path = None
for f in cands:
    try:
        cols = [c.lower() for c in pd.read_csv(f, nrows=1).columns]
    except Exception:
        continue
    if 'outcome' in cols and 'glucose' in cols:   # recognises the Pima file whatever it is called
        path = f; break
if path is None:
    raise FileNotFoundError("No Pima CSV found. Click 'Add Input' (right sidebar) and add 'Pima Indians Diabetes Database', or upload diabetes_pima_real.csv as a dataset and add it.")
df = pd.read_csv(path); df.columns = [c.lower() for c in df.columns]
df = df.rename(columns={'diabetespedigreefunction':'diabetes_pedigree','bloodpressure':'blood_pressure','skinthickness':'skin_thickness'})
print(path, df.shape); df.head()

## 1. Clean: impossible zeros are really missing values

In [ ]:
zero_as_nan = ['glucose','blood_pressure','skin_thickness','insulin','bmi']
df[zero_as_nan] = df[zero_as_nan].replace(0, np.nan)
print(df.isna().sum()); print(df.outcome.value_counts(normalize=True).round(3))

## 2. Split. Imputation lives inside the pipeline, so no leakage.

In [ ]:
X, y = df.drop(columns='outcome'), df.outcome
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=SEED)
models = {
 'logreg': Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('m',LogisticRegression(class_weight='balanced',max_iter=1000))]),
 'rf': Pipeline([('imp',SimpleImputer(strategy='median')),('m',RandomForestClassifier(n_estimators=400,min_samples_leaf=5,class_weight='balanced',random_state=SEED))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for n,m in models.items():
    s = cross_val_score(m, Xtr, ytr, cv=cv, scoring='roc_auc'); print(f'{n}: CV AUC {s.mean():.3f} +/- {s.std():.3f}')

## 3. Evaluate on the held-out test set

In [ ]:
best = models['rf'].fit(Xtr, ytr)          # swap to 'logreg' if it scores better above
p = best.predict_proba(Xte)[:,1]
print('Test AUC:', round(roc_auc_score(yte,p),3))
# choose threshold for recall >= 0.80 (missed cases cost more than false alarms)
pr, rc, th = precision_recall_curve(yte, p)
thr = th[np.where(rc[:-1] >= .80)[0].max()]
print('Threshold for ~80% recall:', round(thr,3))
print(classification_report(yte, (p>=thr).astype(int)))
ConfusionMatrixDisplay.from_predictions(yte, (p>=thr).astype(int)); plt.show()
CalibrationDisplay.from_predictions(yte, p, n_bins=8); plt.show()

## 4. Explain with SHAP

In [ ]:
rf = best.named_steps['m']
Xte_imp = pd.DataFrame(best.named_steps['imp'].transform(Xte), columns=X.columns)
sv = shap.TreeExplainer(rf).shap_values(Xte_imp)
sv = sv[1] if isinstance(sv, list) else (sv[:,:,1] if sv.ndim==3 else sv)
shap.summary_plot(sv, Xte_imp)                         # global
shap.summary_plot(sv, Xte_imp, plot_type='bar')
i = 0
shap.force_plot(shap.TreeExplainer(rf).expected_value[1] if np.ndim(shap.TreeExplainer(rf).expected_value) else shap.TreeExplainer(rf).expected_value,
                sv[i], Xte_imp.iloc[i], matplotlib=True)  # one patient